# f6_m02a_lime.ipynb
**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 6 — Interpretabilidad y Evaluación Final |
| **Módulo** | M02a — LIME |

---

## 🎯 Qué hace

Genera explicaciones locales con LIME (Local Interpretable Model-agnostic Explanations)
sobre los 3 perfiles representativos definidos en m01b (abandono, no abandono, riesgo).
LIME complementa SHAP: mientras SHAP usa valores de Shapley exactos basados en el modelo,
LIME aproxima localmente el comportamiento con un modelo lineal simple.

Modelo: ganador dinámico (leído de `metricas_modelo.json`).

## 📋 Requisitos

- `data/06_evaluacion/metricas_modelo.json` — fuente única de verdad del ganador
- `results/fase6/perfiles_locales.parquet` — generado por f6_m01b
- `results/fase6/shap_importancia_comparativa.parquet` — generado por f6_m01a
- `data/05_modelado/X_test_prep.parquet`
- `data/05_modelado/X_train_prep.parquet` — referencia de distribución para LIME
- `data/05_modelado/y_test.parquet`
- `data/05_modelado/models/<modelo_ganador>.pkl` (dinámico)
- Paquete: lime

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `results/fase6/lime_abandono_real.png` | Gráfico LIME perfil abandono |
| `results/fase6/lime_no_abandono.png` | Gráfico LIME perfil no abandono |
| `results/fase6/lime_zona_riesgo.png` | Gráfico LIME perfil zona riesgo |
| `results/fase6/lime_explicaciones.parquet` | Coeficientes LIME para los 3 perfiles |
| `docs/html/fase6/m02a_lime.html` | Informe HTML |

## 🔄 Flujo

```
metricas_modelo.json → ganador dinámico
perfiles_locales.parquet (m01b) + X_train_prep + modelo ganador
    ↓ Instanciar LimeTabularExplainer con datos de entrenamiento
    ↓ Explicar 3 perfiles (5000 perturbaciones cada uno)
    ↓ Gráficos de barras por perfil
    ↓ Comparativa consenso SHAP vs LIME
    → lime_explicaciones.parquet + docs/html/fase6/m02a_lime.html
```

## ➡️ Siguiente

`f6_m02b_dice.ipynb` — contrafactuales DiCE


In [1]:
# ============================================================
# CELDA 1: CONFIGURACIÓN DE RUTAS
# ROOT detectado subiendo niveles hasta encontrar src/
# ============================================================
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

def _encontrar_root(start: Path) -> Path:
    for parent in [start] + list(start.parents):
        if (parent / 'src').is_dir():
            return parent
    raise FileNotFoundError('No se encontró src/ subiendo desde ' + str(start))

ROOT = _encontrar_root(Path.cwd())
sys.path.insert(0, str(ROOT))

DIR_DATA    = ROOT / 'data' / '05_modelado'
DIR_MODELS  = ROOT / 'data' / '05_modelado' / 'models'
DIR_RESULTS = ROOT / 'results' / 'fase6'
DIR_HTML    = ROOT / 'docs' / 'html' / 'fase6'
DIR_RESULTS.mkdir(parents=True, exist_ok=True)
DIR_HTML.mkdir(parents=True, exist_ok=True)

# Ahora (v2): JSON del ganador dinámico
RUTA_JSON = ROOT / 'data' / '06_evaluacion' / 'metricas_modelo.json'

print(f'ROOT:       {ROOT}')
print(f'DIR_MODELS: {DIR_MODELS}')
print(f'DIR_RESULTS:{DIR_RESULTS}')
print(f'RUTA_JSON:  {RUTA_JSON}')

ROOT:       c:\FF\AU_UJI_v2
DIR_MODELS: c:\FF\AU_UJI_v2\data\05_modelado\models
DIR_RESULTS:c:\FF\AU_UJI_v2\results\fase6
RUTA_JSON:  c:\FF\AU_UJI_v2\data\06_evaluacion\metricas_modelo.json


In [2]:
# ============================================================
# CELDA 2: IMPORTS Y CARGA DEL GANADOR DINÁMICO
# Sistema dinámico (v2): el ganador se lee de metricas_modelo.json
# ============================================================
import json
import base64
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import joblib
from lime import lime_tabular
from src.html.render import render_pagina_desde_fichero
from src.config_entorno import NOMBRES_LEGIBLES_FEATURES

matplotlib.rcParams['figure.dpi'] = 120

def nombre_legible(f: str) -> str:
    return NOMBRES_LEGIBLES_FEATURES.get(f, f.replace('_', ' '))

# Ahora (v2): cargar metadatos del modelo ganador (sistema dinámico)
assert RUTA_JSON.exists(), f'❌ No encontrado: {RUTA_JSON}'
with open(RUTA_JSON, encoding='utf-8') as f:
    meta_json = json.load(f)

nombre_ganador_pkl = meta_json['modelo_pkl']
nombre_ganador     = meta_json['modelo_nombre']
familia_ganador    = meta_json['modelo_familia']

print('Imports OK.')
print(f'Modelo ganador: {nombre_ganador} ({familia_ganador})')
print(f'PKL:            {nombre_ganador_pkl}')

Imports OK.
Modelo ganador: LightGBM (Gradient Boosting)
PKL:            LightGBM__none.pkl


In [3]:
# ============================================================
# CELDA 3: CARGAR DATOS Y MODELO
# X_train_prep: LIME necesita datos de entrenamiento como
# referencia de distribución para generar perturbaciones.
# perfiles_locales: los 3 perfiles definidos en m01b.
# ============================================================
RUTA_PERFILES = DIR_RESULTS / 'perfiles_locales.parquet'
if not RUTA_PERFILES.exists():
    raise FileNotFoundError(
        'No se encontró perfiles_locales.parquet.\n'
        'Ejecuta primero f6_m01b_shap_local.ipynb.'
    )

X_test_prep    = pd.read_parquet(DIR_DATA / 'X_test_prep.parquet')
X_train_prep   = pd.read_parquet(DIR_DATA / 'X_train_prep.parquet')
y_test         = pd.read_parquet(DIR_DATA / 'y_test.parquet').squeeze()
df_perfiles    = pd.read_parquet(RUTA_PERFILES)
modelo_ganador = joblib.load(DIR_MODELS / nombre_ganador_pkl)  # Ahora (v2): dinámico

feature_names = X_test_prep.columns.tolist()

# Detectar features categóricas — LIME necesita saberlo
cat_features = X_test_prep.select_dtypes(include=['object', 'category']).columns.tolist()
cat_idx      = [feature_names.index(f) for f in cat_features]

print(f'Modelo:       {nombre_ganador} ({nombre_ganador_pkl})')
print(f'X_test_prep:  {X_test_prep.shape}')
print(f'X_train_prep: {X_train_prep.shape}')
print(f'Features:     {len(feature_names)} ({len(cat_features)} categóricas)')
print(f'Perfiles:     {df_perfiles["perfil"].tolist()}')

Modelo:       LightGBM (LightGBM__none.pkl)
X_test_prep:  (6725, 27)
X_train_prep: (26896, 27)
Features:     27 (0 categóricas)
Perfiles:     ['abandono_real', 'no_abandono', 'zona_riesgo']


In [4]:
# ============================================================
# CELDA 4: INSTANCIAR LIME EXPLAINER
# LimeTabularExplainer aprende la distribución de cada feature
# desde X_train para generar perturbaciones realistas.
# discretize_continuous=True: discretiza numéricas en intervalos
# para que el modelo lineal local sea más interpretable.
# ============================================================
explainer_lime = lime_tabular.LimeTabularExplainer(
    training_data=X_train_prep.values,
    feature_names=feature_names,
    categorical_features=cat_idx,
    class_names=['No abandona', 'Abandona'],
    mode='classification',
    discretize_continuous=True,
    random_state=42
)

print(f'✅ LIME explainer instanciado para {nombre_ganador}.')
print(f'   Referencia:   {X_train_prep.shape[0]:,} observaciones de entrenamiento')
print(f'   Features:     {len(feature_names)} ({len(cat_idx)} categóricas)')
print(f'   Discretizado: True (intervalos para interpretabilidad)')

✅ LIME explainer instanciado para LightGBM.
   Referencia:   26,896 observaciones de entrenamiento
   Features:     27 (0 categóricas)
   Discretizado: True (intervalos para interpretabilidad)


In [5]:
# ============================================================
# CELDA 5: GENERAR EXPLICACIONES LIME — 3 PERFILES
# 5000 perturbaciones por perfil — estándar LIME.
# Recupera índices posicionales desde perfiles_locales.
# Consistente con SHAP local (m01b): mismos 3 perfiles.
# ============================================================
prob  = modelo_ganador.predict_proba(X_test_prep)[:, 1]
y_arr = y_test.values.ravel()

# El índice de df_perfiles son IDs de alumno; los nombres ('abandono_real',
# 'no_abandono', 'zona_riesgo') están en la columna `perfil`.
idx_posicionales = [X_test_prep.index.get_loc(idx) for idx in df_perfiles.index.tolist()]
perfiles_orden   = df_perfiles['perfil'].tolist()

perfiles_labels = {
    'abandono_real': 'Abandono (VP)',
    'no_abandono':   'No abandona (VN)',
    'zona_riesgo':   'Zona de riesgo',
}

explicaciones = {}
registros     = []

for perfil, idx_pos in zip(perfiles_orden, idx_posicionales):
    label     = perfiles_labels.get(perfil, perfil)
    instancia = X_test_prep.iloc[idx_pos].values
    print(f'Calculando LIME para {label}...')

    exp = explainer_lime.explain_instance(
        data_row=instancia,
        predict_fn=modelo_ganador.predict_proba,  # Ahora (v2): ganador dinámico
        num_features=15,
        num_samples=5000,
        labels=(1,)
    )
    explicaciones[perfil] = exp

    for feat, coef in exp.as_list(label=1):
        registros.append({
            'perfil':    perfil,
            'feature':   feat,
            'coef_lime': coef,
            'prob_real': prob[idx_pos],
            'y_real':    y_arr[idx_pos],
        })
    print(f'  ✅ Prob: {prob[idx_pos]:.3f} | y_real: {y_arr[idx_pos]}')

df_lime = pd.DataFrame(registros)
df_lime.to_parquet(DIR_RESULTS / 'lime_explicaciones.parquet')
print(f'\n✅ lime_explicaciones.parquet guardado ({len(df_lime)} registros).')

Calculando LIME para Abandono (VP)...
  ✅ Prob: 0.992 | y_real: 1
Calculando LIME para No abandona (VN)...
  ✅ Prob: 0.017 | y_real: 0
Calculando LIME para Zona de riesgo...
  ✅ Prob: 0.500 | y_real: 1

✅ lime_explicaciones.parquet guardado (45 registros).


In [6]:
# ============================================================
# CELDA 6: GRÁFICOS LIME — UNO POR PERFIL
# Barras horizontales:
#   rojo  (#dc2626) = empuja hacia abandono (coef positivo)
#   azul  (#1e4d8c) = reduce el riesgo (coef negativo)
# Paleta institucional UJI (alineada con config_app.py, abril 2026).
# ============================================================
COLOR_ABANDONO = '#dc2626'  # COLORES["abandono"] en app
COLOR_PRIMARIO = '#1e4d8c'  # COLORES["primario"] en app

rutas_lime = {}

for perfil, idx_pos in zip(perfiles_orden, idx_posicionales):
    label = perfiles_labels.get(perfil, perfil)
    exp   = explicaciones[perfil]
    ruta  = DIR_RESULTS / f'lime_{perfil}.png'
    rutas_lime[perfil] = ruta

    feats_coefs = exp.as_list(label=1)
    feats = [fc[0] for fc in feats_coefs]
    coefs = [fc[1] for fc in feats_coefs]

    # Ordenar por valor absoluto
    orden = np.argsort(np.abs(coefs))
    feats = [feats[j] for j in orden]
    coefs = [coefs[j] for j in orden]

    colores_barras = [COLOR_ABANDONO if c > 0 else COLOR_PRIMARIO for c in coefs]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(feats, coefs, color=colores_barras, alpha=0.85)
    ax.axvline(0, color='gray', linewidth=0.8)
    ax.set_xlabel('Coeficiente LIME (positivo = hacia abandono)')
    ax.set_title(
        f'Fase 6 — LIME ({nombre_ganador}): {label} (prob={prob[idx_pos]:.3f})',
        fontsize=13, pad=12
    )
    plt.tight_layout()
    plt.savefig(ruta, dpi=120, bbox_inches='tight')
    plt.close()
    print(f'✅ Gráfico guardado: {ruta.name}')

✅ Gráfico guardado: lime_abandono_real.png
✅ Gráfico guardado: lime_no_abandono.png
✅ Gráfico guardado: lime_zona_riesgo.png


In [7]:
# ============================================================
# CELDA 7: COMPARATIVA SHAP vs LIME — VISUAL Y TEXTUAL 🏆
# Features en consenso = hallazgos más robustos.
# Gráfico de barras coloreado: verde=consenso, gris=solo SHAP.
# Detecta automáticamente la columna del modelo ganador.
# Paleta UJI 2026 (alineada con config_app.py).
# ============================================================
COLOR_CONSENSO   = '#10b981'  # COLORES["exito"] en app — verde vivo
COLOR_SOLO_SHAP  = '#94a3b8'  # COLORES["texto_muy_suave"] — gris claro

RUTA_SHAP_IMP = DIR_RESULTS / 'shap_importancia_comparativa.parquet'
consenso      = []
top10_shap    = []
ruta_consenso = None

if RUTA_SHAP_IMP.exists():
    df_shap_imp = pd.read_parquet(RUTA_SHAP_IMP)

    # Detectar columna del ganador (puede llamarse 'LightGBM', 'CatBoost', etc.)
    if nombre_ganador in df_shap_imp.columns:
        col_ganador = nombre_ganador
    else:
        # Fallback: primera columna numérica que no sea rank/discrepancia
        candidatas = [c for c in df_shap_imp.select_dtypes(include=[np.number]).columns
                      if 'rank' not in c.lower() and 'discrep' not in c.lower()]
        col_ganador = candidatas[0] if candidatas else None

    if col_ganador is None:
        print('⚠️  No se detectó columna del ganador en el parquet SHAP.')
    else:
        print(f'Columna SHAP del ganador detectada: {col_ganador}')

        top10_shap = df_shap_imp.nsmallest(10, 'rank_medio').index.tolist()

        lime_abandono = df_lime[df_lime['perfil'] == 'abandono_real'].copy()
        lime_abandono['feature_base'] = lime_abandono['feature'].str.extract(r'^([^<>=!]+)')[0].str.strip()
        top10_lime = lime_abandono.nlargest(10, 'coef_lime')['feature_base'].tolist()

        # Detectar consenso: feature SHAP que aparece en algún label LIME
        consenso = [f for f in top10_shap if any(f in fl for fl in top10_lime)]

        print(f'Top 10 SHAP | Top 10 LIME | Consenso: {len(consenso)}/10')
        for f in top10_shap:
            marca = '✅ CONSENSO' if f in consenso else '  solo SHAP'
            print(f'  {nombre_legible(f):35s} {marca}')

        # Gráfico visual de consenso
        shap_vals = df_shap_imp.loc[top10_shap, col_ganador].values
        colores   = [COLOR_CONSENSO if f in consenso else COLOR_SOLO_SHAP for f in top10_shap]
        labels    = [nombre_legible(f) for f in top10_shap]

        fig, ax = plt.subplots(figsize=(10, 5))
        ax.barh(labels[::-1], shap_vals[::-1], color=colores[::-1], alpha=0.85)
        ax.set_xlabel(f'Importancia SHAP media ({nombre_ganador})')
        ax.set_title(
            f'Fase 6 — Consenso SHAP + LIME: Top 10 features ({nombre_ganador})\n'
            'Verde = aparece también en LIME (hallazgo robusto) · Gris = solo SHAP',
            fontsize=12
        )
        # Leyenda manual
        from matplotlib.patches import Patch
        ax.legend(handles=[
            Patch(color=COLOR_CONSENSO,  label=f'Consenso SHAP+LIME ({len(consenso)} features)'),
            Patch(color=COLOR_SOLO_SHAP, label=f'Solo SHAP ({10-len(consenso)} features)'),
        ], fontsize=9, loc='lower right')
        plt.tight_layout()
        ruta_consenso = DIR_RESULTS / 'lime_consenso_shap.png'
        plt.savefig(ruta_consenso, dpi=120, bbox_inches='tight')
        plt.close()
        print(f'\n✅ Gráfico consenso guardado: {ruta_consenso.name}')
else:
    print('⚠️  shap_importancia_comparativa.parquet no encontrado.')

Columna SHAP del ganador detectada: LightGBM
Top 10 SHAP | Top 10 LIME | Consenso: 4/10
  Créditos superados 1er año          ✅ CONSENSO
  Años trabajando                     ✅ CONSENSO
  Años con beca                       ✅ CONSENSO
  Años sin beca                         solo SHAP
  Créditos repetidos                  ✅ CONSENSO
  Años sin notas                        solo SHAP
  Nota 1er año                          solo SHAP
  Situación laboral                     solo SHAP
  Nota de acceso                        solo SHAP
  Tasa de repetición                    solo SHAP

✅ Gráfico consenso guardado: lime_consenso_shap.png


In [8]:
# ============================================================
# CELDA 8: GENERAR HTML
# render_pagina_desde_fichero — estándar del proyecto.
# Incluye bloque Wilcoxon dinámico para justificar el modelo ganador.
# Paleta UJI 2026 (alineada con config_app.py).
# ============================================================
from src.html.wilcoxon_block import bloque_wilcoxon_html

def img_b64(ruta: Path) -> str:
    if not ruta or not ruta.exists():
        return ''
    with open(ruta, 'rb') as fh:
        return base64.b64encode(fh.read()).decode()

def bloque_imagen(b64: str, titulo: str, caption: str) -> str:
    if not b64:
        return f'<p style="color:#dc2626">⚠️ Imagen no disponible: {titulo}</p>'
    return (
        '<div style="margin:24px 0">'
        f'<h3 style="color:#2d3748;font-size:15px">{titulo}</h3>'
        f'<img src="data:image/png;base64,{b64}" '
        'style="max-width:100%;border-radius:6px;box-shadow:0 2px 8px rgba(0,0,0,.1)">'
        f'<p style="color:#718096;font-size:12px;margin-top:6px">{caption}</p>'
        '</div>'
    )

contenido = (
    f'<h2 style="color:#2d3748">Fase 6 — M02a · LIME: Explicaciones Locales Alternativas</h2>'
    + bloque_wilcoxon_html(ROOT, nombre_ganador)
    + '<p style="color:#4a5568;font-size:14px;max-width:800px">'
    'LIME (<em>Local Interpretable Model-agnostic Explanations</em>) aproxima el comportamiento '
    f'del modelo <strong>{nombre_ganador}</strong> localmente con un modelo lineal simple. '
    'A diferencia de SHAP, que usa valores de Shapley exactos, LIME genera perturbaciones '
    'de la instancia y ajusta un modelo lineal en esa región. Ambos métodos son complementarios: '
    'el consenso entre SHAP y LIME identifica las variables más robustamente importantes.'
    '</p>'
    + bloque_imagen(
        img_b64(rutas_lime['abandono_real']),
        'LIME — Alumno que abandona (verdadero positivo)',
        'Barras rojas: factores que el modelo lineal local identifica como impulsores del abandono. '
        'Barras azules: factores protectores. '
        'Los valores son coeficientes del modelo lineal local ajustado por LIME.')
    + bloque_imagen(
        img_b64(rutas_lime['no_abandono']),
        'LIME — Alumno que no abandona (verdadero negativo)',
        'Las barras azules dominan: LIME identifica factores protectores '
        'que reducen el riesgo predicho.')
    + bloque_imagen(
        img_b64(rutas_lime['zona_riesgo']),
        'LIME — Alumno en zona de riesgo (probabilidad ~0.5)',
        'Factores de riesgo y protectores se equilibran. '
        'Este perfil es el más sensible a pequeños cambios en las variables.')
    + (bloque_imagen(
        img_b64(ruta_consenso),
        '🏆 Consenso SHAP + LIME — Features más robustas',
        'Verde = features que aparecen en el top 10 de SHAP Y en las más importantes de LIME. '
        'Estas son las variables más robustamente importantes del modelo: '
        'dos métodos completamente distintos llegan a la misma conclusión.'
      ) if ruta_consenso else '')
    + '<div style="margin-top:24px;padding:16px;background:#ebf8ff;'
      'border-left:4px solid #1e4d8c;border-radius:6px;font-size:13px;color:#2c5282">'
      '<strong>SHAP vs LIME:</strong> SHAP calcula contribuciones exactas basadas en la '
      'teoría de juegos (valores de Shapley). LIME aproxima localmente con un modelo lineal. '
      'Cuando ambos métodos coinciden en las variables más importantes, '
      'el hallazgo es especialmente robusto y defendible ante el tribunal.'
      '</div>'
)

html_completo = render_pagina_desde_fichero('f6_m02a_lime.ipynb', contenido)
ruta_html = DIR_HTML / 'm02a_lime.html'
ruta_html.write_text(html_completo, encoding='utf-8')
print(f'✅ HTML generado: {ruta_html}')

✅ HTML generado: c:\FF\AU_UJI_v2\docs\html\fase6\m02a_lime.html
